Ставим зависимости

In [ ]:
!pip install datasets evaluate seqeval -q

## Загружаем и готовим датасет

In [2]:
import torch
import os
from datasets import load_dataset

raw_datasets = load_dataset("conll2003", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [3]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

## Готовим модели

In [ ]:
from transformers import AutoTokenizer

teacher_checkpoint_name = "dslim/bert-large-NER"
student_checkpoint_name = "dslim/bert-base-NER"

t_tokenizer = AutoTokenizer.from_pretrained(teacher_checkpoint_name)
s_tokenizer = AutoTokenizer.from_pretrained(student_checkpoint_name)

assert t_tokenizer.vocab == s_tokenizer.vocab

In [5]:
tokenizer = t_tokenizer


def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels


def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [6]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

ner_feature = raw_datasets["train"].features["ner_tags"]

label_names = ner_feature.feature.names

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [7]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader


data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)

eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

test_dataloader = DataLoader(
    tokenized_datasets["test"], collate_fn=data_collator, batch_size=8
)

In [8]:
id2label = {
    0: "O",
    1: "B-MISC",
    2: "I-MISC",
    3: "B-PER",
    4: "I-PER",
    5: "B-ORG",
    6: "I-ORG",
    7: "B-LOC",
    8: "I-LOC"
}

def postprocess(predictions, labels):
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

Грузим модель

In [19]:
from transformers import AutoModelForTokenClassification

device = 'cuda'


teacher_model = AutoModelForTokenClassification.from_pretrained(
    teacher_checkpoint_name,
    device_map=device
)

student_model = AutoModelForTokenClassification.from_pretrained(
    student_checkpoint_name,
    device_map=device
)

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. in

Ф-ция для нахождения метрик модели

In [10]:
import evaluate
import torch
from tqdm import tqdm

def evaluate_bert_ner(eval_dataloader, model, device):
    metric = evaluate.load("seqeval")
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            batch = {k: v.to(device)for k, v in batch.items()}
            outputs = model(**batch)

        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]

        true_predictions, true_labels = postprocess(predictions, labels)
        metric.add_batch(predictions=true_predictions, references=true_labels)

    results = metric.compute()
    return results['overall_f1']

Ф-ция для нахождения среднего времени работы

In [11]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import time
from tqdm import tqdm


# Среднее время работы
def measure_time(nlp, raw_datasets, index, n):
    tsum = 0
    for i in tqdm(range(n)):
        start = time.perf_counter()
        ner_results = nlp(' '.join(raw_datasets["test"][index]["tokens"]))
        tsum += (time.perf_counter() - start)
    return tsum / n

## Дистилляция

Посмотрим на начальные метрики

In [12]:
teacher_f1 = evaluate_bert_ner(eval_dataloader, teacher_model, device)
print('Teacher F1:', teacher_f1)

100%|██████████| 407/407 [00:26<00:00, 15.15it/s]


Teacher F1: 0.8276193527715475


In [13]:
student_f1 = evaluate_bert_ner(eval_dataloader, student_model, device)
print('Student F1:', student_f1)

100%|██████████| 407/407 [00:08<00:00, 48.54it/s]


Student F1: 0.6839153518283106


Обучаем!

In [20]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(student_model.parameters(), lr=2e-5)
epochs = 1
T=2
soft_target_loss_weight = 0.25

In [21]:
from tqdm import tqdm

teacher_model.eval()

best_model_state = None
best_f1 = -1

for epoch in range(epochs):
    student_model.train()
    train_loss = 0

    for batch in tqdm(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        with torch.no_grad():
            teacher_logits = teacher_model(**batch).logits

        student_logits = student_model(**batch).logits

        soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
        soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)

        # soft_targets_loss = -torch.sum(soft_targets * soft_prob) / soft_prob.size()[0] * (T**2)
        soft_targets_loss = torch.sum(
            soft_targets * (soft_targets.log() - soft_prob)
        ) / soft_prob.size()[0] * (T**2)

        label_loss = criterion(student_logits.permute(0, 2, 1), batch['labels'])

        loss = soft_target_loss_weight * soft_targets_loss + (1 - soft_target_loss_weight) * label_loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    student_model.eval()

    student_f1 = evaluate_bert_ner(eval_dataloader, student_model, device)
    print('Student F1:', student_f1)

    if student_f1 > best_f1:
        best_model_state = student_model.state_dict()
        best_f1 = student_f1

    print(f'Epoch: {epoch}, loss: {loss / len(train_dataloader):.3f} | Student F1: {student_f1:.3f}')

100%|██████████| 407/407 [00:12<00:00, 33.09it/s]


Student F1: 0.8696999347684279
Epoch: 0, loss: 0.000 | Student F1: 0.870


Посмотрим на финальные метрики

In [22]:
student_model.eval()

student_f1 = evaluate_bert_ner(eval_dataloader, student_model, device)
print('Student F1:', student_f1)

100%|██████████| 407/407 [00:09<00:00, 43.96it/s]


Student F1: 0.8696999347684279


Замеряем скорость инференса

In [23]:
nlp = pipeline("ner", model=student_model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cuda


In [28]:
seconds_mean_time_cuda = measure_time(nlp, raw_datasets, 3, 100)
print()
print('Mean time on cuda:', seconds_mean_time_cuda*1000, "ms")

100%|██████████| 100/100 [00:01<00:00, 86.35it/s]


Mean time on cuda: 11.511904439987575 ms


Замерим время на CPU.

Из-за особенностей, мы не можем просто сменить device у student_model, поэтому просто подгрузим аналогичную модель и замерим время инференса у нее.

In [29]:
student_model_cpu = AutoModelForTokenClassification.from_pretrained(
    student_checkpoint_name,
    device_map='cpu'
)

nlp = pipeline("ner", model=student_model_cpu, tokenizer=tokenizer, aggregation_strategy="simple")

seconds_mean_time_cpu = measure_time(nlp, raw_datasets, 3, 100)
print()
print('Mean time on cuda:', seconds_mean_time_cpu*1000, "ms")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
100%|██████████| 100/100 [00:16<00:00,  6.15it/s]


Mean time on cuda: 160.4926370299927 ms
